# Example 12.1 / 4: Let the Solver Run the Passes

Companion notebook for **12.1 Multiple Objectives** in the AMPL lab Documentation, section "Let the Solver Run the Passes". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Let the Solver Run the Passes

MP solver drivers can run the same passes for you. Give each objective a priority with the
`.objpriority` suffix (higher is optimized first), allow the protected objective to worsen with
`.objabstol`, and turn on multiobjective mode with the solver option `obj:multi=2`. The suffix
values are assigned with `ampl.eval("let ...")` because amplpy has no dedicated method for a
user-declared objective suffix; `solve(..., highs_options=...)` passes the option and
`ampl.solve_result` reports the status.

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
set PLANTS;                           # production plants
param unit_cost {PLANTS} >= 0;        # dollars per unit
param unit_emission {PLANTS} >= 0;    # kg CO2 per unit
param capacity {PLANTS} >= 0;         # units
param demand >= 0;                    # units

var Make {p in PLANTS} >= 0, <= capacity[p];  # units produced at each plant

# objective 1: production cost, dollars
minimize TotalCost: sum {p in PLANTS} unit_cost[p] * Make[p];
# objective 2: emissions, kg CO2
minimize TotalEmission: sum {p in PLANTS} unit_emission[p] * Make[p];

subject to MeetDemand: sum {p in PLANTS} Make[p] = demand;  # total output covers demand exactly

suffix objpriority IN, integer;       # solve order: higher value is optimized first
suffix objweight IN;                  # blend weight among objectives of equal priority
suffix objabstol IN;                  # allowed absolute worsening, objective units
suffix objreltol IN;                  # allowed relative worsening, fraction

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [4]:
# data.py
ampl.set["PLANTS"] = ["A", "B"]
ampl.param["unit_cost"] = {"A": 2, "B": 4}
ampl.param["unit_emission"] = {"A": 5, "B": 1}
ampl.param["capacity"] = {"A": 8, "B": 8}
ampl.param["demand"] = 10

In [5]:
# run.py
# Cost first, but allow it to rise by 4 dollars to reduce emissions.
ampl.eval("let TotalCost.objpriority := 2; let TotalEmission.objpriority := 1;")
ampl.eval("let TotalCost.objabstol := 4;")
ampl.solve(solver="highs", highs_options="obj:multi=2")
print(ampl.solve_result, ampl.var["Make"].to_dict())
print("cost", ampl.obj["TotalCost"].value(), "emission", ampl.obj["TotalEmission"].value())

HiGHS 1.14.0:   obj:multi = 2
HiGHS 1.14.0: optimal solution; objective 28
Individual objective values:
	_sobj[1] = 28
	_sobj[2] = 34
1 simplex iterations
0 barrier iterations
Objective = TotalCost
solved {'A': 6, 'B': 4}
cost 28.0 emission 34.0


### Pure AMPL: data.dat and model.run

The same in pure AMPL, with the same `model.mod`:

In [6]:
%%writefile data.dat
set PLANTS := A B;
param: unit_cost unit_emission capacity :=
    A    2   5   8
    B    4   1   8 ;
param demand := 10;

Writing data.dat


In [7]:
%%ampl_eval
reset;
reset options;
model model.mod;
data data.dat;

In [8]:
%%ampl_eval
# model.run
let TotalCost.objpriority := 2;
let TotalEmission.objpriority := 1;
let TotalCost.objabstol := 4;
option solver highs;
option highs_options 'obj:multi=2';
solve;
display Make, TotalCost, TotalEmission;

HiGHS 1.14.0:   obj:multi = 2
HiGHS 1.14.0: optimal solution; objective 28
Individual objective values:
	_sobj[1] = 28
	_sobj[2] = 34
1 simplex iterations
0 barrier iterations
Objective = TotalCost
Make [*] :=
A  6
B  4
;

TotalCost = 28
TotalEmission = 34

